# Why the Bayes accuracy takes off: KL and Jensen-Shannon divergences of `P(M | Q)`

Issue #270. In the learnability setup of Barratt et al., PRL 129, 200602 (2022) (notebooks `benchmarks/01` and `02`), an observer sees the measurement record `M` of a monitored U(1) brickwork and guesses which of two charges `Q+ = N // 2` or `Q- = N // 2 - 1` the chain started in. Whatever the classifier, how well it can do is set by how far apart the two conditional laws `P+(M) = P(M | Q+)` and `P-(M) = P(M | Q-)` are. This notebook measures that distance against the measurement probability `p`, next to the Bayes accuracy (the "blue curve"):

- the **Kullback-Leibler divergence** `D(P+ || P-) = E_{M ~ P+}[ln P+(M) / P-(M)]`, estimated by Monte Carlo on records drawn with their Born probability. It is **infinite at every `p > 0`**: a record that measures enough sites in one round can be impossible under one charge. So it is split into the **infinite mass** `pi_inf` (probability that a record gives the charge for certain) and the **finite part** (the KL between the two laws restricted to the records that both allow), with the Jeffreys distance `D_sym`, the mean of the two directions;
- the **Jensen-Shannon divergence**, which is the mutual information `I(Q; M)` (in bits, between 0 and 1), always finite, and equal to `1 - E[H(P(Q | M))]` with the mean entropy of Fig. 2c;
- the **Bayes error** `P_e` and its bounds from JS, Bhattacharyya and Le Cam.

Two observers: one who does not know the gates (Eve of Barratt et al., likelihood averaged over the Haar gates, the blue curve), and one who knows the circuit (likelihood from the state vector, averaged over `N_c` circuits with the spread between circuits).

Library: `QELM.evaluation.kl_divergence`. Experiment: `experiments/kl_divergence/` (config `conf/kl_divergence.yaml`, report in its `README.md`).

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here or through the environment variable of the same name (what the test `tests/test_kl_divergence_notebook.py` does for a small sweep). The defaults are those of the issue: `N = 6, 8, 10`, `T = N` time steps (`2 T` measurement rounds), `p` from 0.05 to 0.40, `L = 8000` records for the observer who does not know the gates, `N_c = 20` circuits of `PER_CIRCUIT = 1000` records for the one who knows them. `N = 10` takes about 4 minutes per `p` on one core; `JOBS` runs several `p` at once (on a cluster, run `experiments/kl_divergence/kl_divergence.sh` directly and set `RUN = 0` here to read back its sweep, copied under `outputs/kl_divergence_notebook/`). `P_C` is the dashed line of the figures and the origin of the fit `D ~ (p - p_c)**beta`. `ENCODING=local` replaces the charge by the Z outcome of a reference qubit in a Bell pair with the middle site of the Néel state, and `GATES=u4` the U(1) gates by Haar U(4) gates (README, "Local encoding").

In [ ]:
L = os.environ.get("L", "8000")  # records, gates unknown (samples.L)
N_C = os.environ.get("N_C", "20")  # circuits, gates known
PER_CIRCUIT = os.environ.get("PER_CIRCUIT", "1000")  # records per circuit
N_VALUES = os.environ.get("N_VALUES", "6 8 10")
SEEDS = os.environ.get("SEEDS", "0")
P_VALUES = os.environ.get(
    "P_VALUES",
    "0.05 0.075 0.1 0.125 0.15 0.175 0.2 0.225 0.25 0.275 0.3 0.325 0.35 0.375 0.4",
)
JOBS = os.environ.get("JOBS", str(os.cpu_count() or 1))
P_C = float(os.environ.get("P_C", "0.175"))
# charge (Barratt et al.) or local (reference qubit on the middle site)
ENCODING = os.environ.get("ENCODING", "charge")
GATES = os.environ.get("GATES", "u1")  # u1 or u4 (Haar U(4))
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Launch the sweep

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151).
EXPERIMENT = "kl_divergence_notebook"

if RUN:
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "L": L,
        "N_C": N_C,
        "PER_CIRCUIT": PER_CIRCUIT,
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "JOBS": JOBS,
        "P_C": str(P_C),
        "ENCODING": ENCODING,
        "GATES": GATES,
        "OUT_DIR": str(sweep_dir),
    }
    n_runs = len(N_VALUES.split()) * len(SEEDS.split()) * len(P_VALUES.split())
    run_with_progress(
        ["bash", "experiments/kl_divergence/kl_divergence.sh"], total=n_runs, env=env
    )
else:
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the metrics

One row per `(N, p)`, averaged over the seeds. `gates unknown` is the Haar-averaged observer (`sep_*` in `metrics.json`), `gates known` the mean over the `N_c` circuits (`circuit_*`). `D_KL` is infinite at every `p > 0`; the table gives its finite part `D_sym` (Jeffreys, nats) and the infinite mass `pi_inf`.

In [ ]:
import importlib.util

# The plotting script of the experiment is not a package: import it by path.
spec = importlib.util.spec_from_file_location(
    "plot_kl_divergence", "experiments/kl_divergence/plot_kl_divergence.py"
)
plot_kl = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_kl)

table = plot_kl.load_summary(sweep_dir)
COLUMNS = [
    ("sep_bayes_accuracy", "acc"),
    ("sep_js_bits", "I(Q;M)"),
    ("sep_jeffreys_finite", "D_sym"),
    ("sep_infinite_mass", "pi_inf"),
    ("circuit_bayes_accuracy", "acc|U"),
    ("circuit_js_bits", "I|U"),
    ("circuit_jeffreys_finite", "D_sym|U"),
]
print(f"{'N':>3} {'p':>6} " + " ".join(f"{name:>9}" for _, name in COLUMNS))
for r in table:
    values = " ".join(f"{r[f'{m}_mean']:>9.4f}" for m, _ in COLUMNS)
    print(f"{r['N']:>3} {r['p']:>6.3f} {values}")

## Bayes accuracy and the divergences that drive it

Top: the Bayes accuracy `1 - P_e` (blue: gates unknown, the blue curve; orange: gates known, shaded spread between circuits). Bottom, log scale: finite Jeffreys distance, `I(Q; M)` and infinite mass. The dashed orange line is `p_c`.

In [ ]:
import matplotlib.pyplot as plt

fits = plot_kl.plot_sweep(sweep_dir, P_C)  # writes the figures and scaling.json
fig, _ = plot_kl.draw_kl_vs_accuracy(table, P_C)
plt.show()

## Finite-size analysis

(a) `I(Q; M)` and (b) `pi_inf` against `p` for every `N`; (c) the rate `alpha` of the fit `I(Q; M) ~ 2**(-alpha N)` at each `p` (a volume-law phase that hides the charge would give `alpha > 0` below `p_c`); (d) the finite Jeffreys distance against `p - p_c`, log-log, with the fit `D ~ (p - p_c)**beta`.

In [ ]:
fig, _, _ = plot_kl.draw_finite_size(table, P_C)
plt.show()
for prefix, rows in fits["alpha"].items():
    print(prefix, " ".join(f"p={r['p']:.3g}: {r['alpha']:+.3f}" for r in rows))
for r in fits["beta"]:
    print(f"beta(N={r['N']}) = {r['beta']:.3f} +- {r['beta_err']:.2g}")

## The Bayes error and its bounds

The Le Cam (Bretagnolle-Huber) bound `P_e >= exp(-D_KL) / 4` of the issue is 0 at every `p > 0`, since `D_KL` is infinite. The bounds from Jensen-Shannon (Lin 1991) and from the Bhattacharyya coefficient stay informative and bracket `P_e`.

In [ ]:
fig, _ = plot_kl.draw_bounds(table, P_C)
plt.show()

## The distribution of the log-likelihood ratio

The divergences are moments of the law of `l(M) = ln P+(M) / P-(M)`. Its histograms per charge, for the largest `N`, at three values of `p` (records of the observer who does not know the gates): the two laws overlap at small `p` and move apart as `p` grows, while a growing fraction of records sits at `l = +-inf` (counted in the legend).

In [ ]:
import numpy as np

from QELM.utils.plots import COLORS, subplots

N_max = max(r["N"] for r in table)
runs = sorted(
    (r for r in plot_kl.load_sweep(sweep_dir, keys=plot_kl.KEYS) if r["N"] == N_max),
    key=lambda r: r["p"],
)
shown = [runs[0], runs[len(runs) // 2], runs[-1]]
fig, axes = subplots(1, len(shown), figsize=(5 * len(shown), 4), squeeze=False)
for ax, row in zip(axes[0], shown):
    data = np.load(sweep_dir / row["run"] / "kl_divergence.npz")
    llr, charge = data["llr"], data["charge"]
    q_plus, q_minus = row["charges"]
    finite = np.isfinite(llr)
    edges = np.histogram_bin_edges(llr[finite], bins=50) if finite.any() else 10
    for q, color, name in ((q_plus, COLORS[0], "Q+"), (q_minus, COLORS[1], "Q-")):
        sel = charge == q
        n_inf = int(np.count_nonzero(~finite & sel))
        ax.hist(
            llr[finite & sel],
            bins=edges,
            color=color,
            alpha=0.5,
            density=True,
            label=f"{name} = {q} ({n_inf} at +-inf)",
        )
    ax.set_title(f"N = {N_max}, p = {row['p']:g}")
    ax.set_xlabel(r"$\ln P_+(M) / P_-(M)$")
    ax.legend(fontsize=9)
axes[0, 0].set_ylabel("density")
plt.show()